# 🏨 Iceberg Property Reviews — Explorer Notebook

**Uses**: `pyspark.sql` → Iceberg catalog → `property_reviews` table  
**Catalog**: Hadoop FileSystem (`./warehouse`)  
**Table**: `local.property_db.property_reviews`  
**Partition**: `country_code` × `review_year`

## 0 · Setup — SparkSession with Iceberg

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Adjust WAREHOUSE to your actual path if running from a different directory
NOTEBOOK_DIR = os.path.abspath('')
PROJECT_DIR  = os.path.dirname(NOTEBOOK_DIR)   # one level up from notebooks/
WAREHOUSE    = os.path.join(PROJECT_DIR, 'warehouse')

CATALOG    = 'local'
FULL_TABLE = 'local.property_db.property_reviews'

spark = (
    SparkSession.builder
    .appName('IcebergViewer')
    .master('local[*]')
    .config(
        'spark.sql.extensions',
        'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions'
    )
    .config(f'spark.sql.catalog.{CATALOG}',
            'org.apache.iceberg.spark.SparkCatalog')
    .config(f'spark.sql.catalog.{CATALOG}.type', 'hadoop')
    .config(f'spark.sql.catalog.{CATALOG}.warehouse', WAREHOUSE)
    .config(
        'spark.jars.packages',
        'org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.4.2'
    )
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('✅ SparkSession ready — Iceberg catalog:', CATALOG)
print('   Warehouse:', WAREHOUSE)

26/03/16 13:19:31 WARN Utils: Your hostname, w3e39 resolves to a loopback address: 127.0.1.1; using 192.168.0.227 instead (on interface enp2s0)
26/03/16 13:19:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/w3e39/Downloads/property_pipeline/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/w3e39/.ivy2/cache
The jars for the packages stored in: /home/w3e39/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.4_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c7ffa7ea-205f-46cb-8478-000d8eabe19b;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.4_2.12;1.4.2 in central
:: resolution report :: resolve 130ms :: artifacts dl 15ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.4_2.12;1.4.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spa

✅ SparkSession ready — Iceberg catalog: local
   Warehouse: /home/w3e39/Downloads/warehouse


## 1 · Schema & Row Count

In [2]:
# Load the entire Iceberg table
df = spark.table(FULL_TABLE)

print(f'Total rows : {df.count()}')
print(f'Columns    : {len(df.columns)}')
print('\nSchema:')
df.printSchema()

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `local`.`property_db`.`property_reviews` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.;
'UnresolvedRelation [local, property_db, property_reviews], [], false


26/03/16 13:19:51 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


## 2 · Sample Rows

In [ ]:
# Register as a temp view so we can use plain SQL
df.createOrReplaceTempView('property_reviews')

spark.sql("""
    SELECT property_id, property_name, country_code,
           star_rating, review_score, review_year,
           reviewer_name, review_individual_score, data_quality_flag
    FROM   property_reviews
    LIMIT  10
""").show(truncate=False)

## 3 · Reviews per Country (partition stats)

In [ ]:
spark.sql("""
    SELECT  country_code,
            COUNT(DISTINCT property_id)    AS properties,
            COUNT(review_id)               AS total_reviews,
            ROUND(AVG(review_individual_score), 2) AS avg_review_score,
            ROUND(AVG(star_rating), 2)     AS avg_stars
    FROM    property_reviews
    GROUP BY country_code
    ORDER BY total_reviews DESC
""").show(40, truncate=False)

## 4 · Review Volume by Year (second partition)

In [ ]:
spark.sql("""
    SELECT  review_year,
            COUNT(*)                         AS total_reviews,
            ROUND(AVG(review_individual_score), 2) AS avg_score,
            COUNT(DISTINCT country_code)     AS countries_active
    FROM    property_reviews
    WHERE   review_year IS NOT NULL
    GROUP BY review_year
    ORDER BY review_year
""").show()

## 5 · Top 10 Properties by Average Review Score

In [ ]:
spark.sql("""
    SELECT  property_id,
            FIRST(property_name)                       AS name,
            FIRST(country_code)                        AS country,
            FIRST(star_rating)                         AS stars,
            ROUND(AVG(review_individual_score), 2)     AS avg_score,
            COUNT(review_id)                           AS num_reviews
    FROM    property_reviews
    WHERE   review_id IS NOT NULL
    GROUP BY property_id
    HAVING  num_reviews >= 2
    ORDER BY avg_score DESC
    LIMIT   10
""").show(truncate=False)

## 6 · Data Quality Distribution

In [ ]:
spark.sql("""
    SELECT  data_quality_flag,
            COUNT(DISTINCT property_id) AS properties,
            COUNT(*)                    AS rows
    FROM    property_reviews
    GROUP BY data_quality_flag
""").show()

## 7 · Reviewer Travel Purpose Breakdown

In [ ]:
spark.sql("""
    SELECT  reviewer_travel_purpose,
            COUNT(*)                            AS reviews,
            ROUND(AVG(review_individual_score),2) AS avg_score
    FROM    property_reviews
    WHERE   reviewer_travel_purpose IS NOT NULL
    GROUP BY reviewer_travel_purpose
    ORDER BY reviews DESC
""").show()

## 8 · Iceberg Metadata — Snapshots & Partitions

In [ ]:
print('=== Snapshots ===')
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM   local.property_db.property_reviews.snapshots
""").show(truncate=False)

print('=== Partition summary ===')
spark.sql("""
    SELECT partition, record_count, file_count
    FROM   local.property_db.property_reviews.partitions
    ORDER BY partition
""").show(50, truncate=False)

## 9 · Partition Pruning Demo (fast scan)

In [ ]:
# Only scans country_code='gb' AND review_year=2024 partitions
spark.sql("""
    SELECT  property_name, reviewer_name,
            review_date, review_individual_score, review_summary
    FROM    property_reviews
    WHERE   country_code = 'GB'
      AND   review_year  = 2024
    ORDER BY review_individual_score DESC
""").show(truncate=False)

## 10 · Reviewer Type vs Star Rating Correlation

In [ ]:
spark.sql("""
    SELECT  reviewer_type,
            ROUND(AVG(star_rating), 2)             AS avg_stars,
            ROUND(AVG(review_individual_score), 2) AS avg_review_score,
            COUNT(*)                               AS total_reviews
    FROM    property_reviews
    WHERE   reviewer_type IS NOT NULL
    GROUP BY reviewer_type
    ORDER BY avg_review_score DESC
""").show(truncate=False)

## 11 · Matplotlib Charts (optional)

In [ ]:
try:
    import matplotlib.pyplot as plt
    import pandas as pd

    # ── Chart 1: Reviews per country ─────────────────────────────────────────
    country_pd = spark.sql("""
        SELECT country_code, COUNT(review_id) AS reviews
        FROM   property_reviews
        WHERE  review_id IS NOT NULL
        GROUP BY country_code
        ORDER BY reviews DESC
        LIMIT 15
    """).toPandas()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].bar(country_pd['country_code'], country_pd['reviews'], color='steelblue')
    axes[0].set_title('Reviews per Country (top 15)')
    axes[0].set_xlabel('Country Code')
    axes[0].set_ylabel('Number of Reviews')
    axes[0].tick_params(axis='x', rotation=45)

    # ── Chart 2: Avg score per year ───────────────────────────────────────────
    year_pd = spark.sql("""
        SELECT  review_year,
                ROUND(AVG(review_individual_score), 2) AS avg_score
        FROM    property_reviews
        WHERE   review_year IS NOT NULL
        GROUP BY review_year
        ORDER BY review_year
    """).toPandas()

    axes[1].plot(year_pd['review_year'], year_pd['avg_score'],
                 marker='o', linewidth=2, color='darkorange')
    axes[1].set_title('Average Review Score by Year')
    axes[1].set_xlabel('Year')
    axes[1].set_ylabel('Avg Score')
    axes[1].set_ylim(0, 10)
    axes[1].grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(os.path.join(PROJECT_DIR, 'notebooks', 'iceberg_charts.png'),
                dpi=120, bbox_inches='tight')
    plt.show()
    print('Charts saved → notebooks/iceberg_charts.png')

except ImportError:
    print('matplotlib not installed — skipping charts (pip install matplotlib pandas)')

## 12 · Clean Up

In [ ]:
spark.stop()
print('SparkSession stopped.')